# Generación de Embeddings para RAG — TFG BTC

**Objetivo:** Generar embeddings semánticos para el corpus de noticias usando `all-MiniLM-L6-v2` (384 dim)  
con GPU de Google Colab, y exportar a CSV para importar en local via `03_setup_rag.py --import`.

**Flujo:**
1. Conectar a PostgreSQL (ngrok tunnel o exportar raw_texts a CSV)
2. Cargar modelo SentenceTransformer en GPU
3. Generar embeddings por lotes
4. Exportar `embeddings_output.csv`
5. Descargar e importar en local: `python agente_ia/03_setup_rag.py --import embeddings_output.csv`

> **Nota:** Este notebook usa la GPU gratuita de Colab T4.  
> Con ~93k noticias tarda ~8 minutos en T4 vs ~2h en CPU.

In [ ]:
# Celda 1 — Instalar dependencias
!pip install -q sentence-transformers sqlalchemy psycopg2-binary pandas

In [ ]:
# Celda 2 — Configuración
# OPCIÓN A: Conexión directa a PostgreSQL (requiere ngrok o IP pública)
# OPCIÓN B: Subir raw_texts.csv manualmente (más seguro)

USE_CSV_INPUT = True  # True = usar CSV; False = conectar a PostgreSQL

# Si USE_CSV_INPUT = False, configurar aquí:
DB_HOST     = 'YOUR_NGROK_HOST'  # ej. '0.tcp.ngrok.io'
DB_PORT     = 'YOUR_NGROK_PORT'  # ej. '12345'
DB_NAME     = 'btc_tfg'
DB_USER     = 'tfg_user'
DB_PASSWORD = 'tfg_password_2026'

BATCH_SIZE  = 512   # Tamaño de lote para GPU
LIMIT       = 0     # 0 = todos los textos
OUTPUT_CSV  = 'embeddings_output.csv'

print(f'Modo: {"CSV" if USE_CSV_INPUT else "PostgreSQL"}')
print(f'Batch size: {BATCH_SIZE}')

In [ ]:
# Celda 3 — Verificar GPU
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Dispositivo: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  Sin GPU — la generación será lenta (~2h para 93k textos)')

In [ ]:
# Celda 4 — Cargar textos
import pandas as pd

if USE_CSV_INPUT:
    # Subir raw_texts.csv desde local:
    # En local: psql btc_tfg -c "COPY (SELECT id, text FROM raw_texts WHERE asset='BTC') TO '/tmp/raw_texts.csv' CSV HEADER"
    from google.colab import files
    print('Sube el archivo raw_texts.csv (id, text):')
    uploaded = files.upload()
    csv_name = list(uploaded.keys())[0]
    df = pd.read_csv(csv_name)
    print(f'Cargados {len(df)} textos desde {csv_name}')
else:
    from sqlalchemy import create_engine, text
    engine = create_engine(
        f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
    )
    limit_clause = f'LIMIT {LIMIT}' if LIMIT > 0 else ''
    df = pd.read_sql(
        f"SELECT id, text FROM raw_texts WHERE asset = 'BTC' ORDER BY id {limit_clause}",
        engine
    )
    print(f'Cargados {len(df)} textos desde PostgreSQL')

print(f'Columnas: {list(df.columns)}')
df.head(2)

In [ ]:
# Celda 5 — Cargar modelo SentenceTransformer
from sentence_transformers import SentenceTransformer
import time

print('Cargando all-MiniLM-L6-v2 (384 dim)...')
t0 = time.time()
model = SentenceTransformer('all-MiniLM-L6-v2', device=device)
print(f'Modelo cargado en {time.time()-t0:.1f}s')

# Test rápido
test_emb = model.encode('Bitcoin price rally')
print(f'Dimensión de embedding: {len(test_emb)}')

In [ ]:
# Celda 6 — Generar embeddings
import numpy as np
import json
import time

texts = df['text'].fillna('').tolist()
ids   = df['id'].tolist()

print(f'Generando embeddings para {len(texts)} textos (batch_size={BATCH_SIZE})...')
t0 = time.time()

all_embeddings = model.encode(
    texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,  # L2 normalize para cosine similarity
)

elapsed = time.time() - t0
print(f'Embeddings generados: {all_embeddings.shape}')
print(f'Tiempo: {elapsed:.1f}s ({elapsed/len(texts)*1000:.1f}ms por texto)')

In [ ]:
# Celda 7 — Exportar a CSV
import json

rows = []
for raw_id, emb in zip(ids, all_embeddings):
    rows.append({
        'raw_text_id': int(raw_id),
        'embedding':   json.dumps(emb.tolist()),
    })

df_out = pd.DataFrame(rows)
df_out.to_csv(OUTPUT_CSV, index=False)
print(f'Exportado: {OUTPUT_CSV} ({len(df_out)} filas)')

# Tamaño del archivo
import os
size_mb = os.path.getsize(OUTPUT_CSV) / 1e6
print(f'Tamaño: {size_mb:.1f} MB')
df_out.head(2)

In [ ]:
# Celda 8 — Descargar CSV
from google.colab import files
files.download(OUTPUT_CSV)
print(f'Descargando {OUTPUT_CSV}...')
print()
print('Siguiente paso (en local):')
print(f'  python agente_ia/03_setup_rag.py --import {OUTPUT_CSV}')

In [ ]:
# Celda 9 (opcional) — Validación: búsqueda semántica de ejemplo
from sklearn.metrics.pairwise import cosine_similarity

QUERY = 'Bitcoin ETF SEC approval'
query_emb = model.encode([QUERY], normalize_embeddings=True)

# Buscar los 5 más similares en la muestra
sample_size = min(10000, len(all_embeddings))
sims = cosine_similarity(query_emb, all_embeddings[:sample_size])[0]
top_k = sims.argsort()[-5:][::-1]

print(f'Query: "{QUERY}"')
print(f'Top 5 resultados (muestra de {sample_size} textos):')
for i, idx in enumerate(top_k, 1):
    text_preview = texts[idx][:120].replace('\n', ' ')
    print(f'  [{i}] sim={sims[idx]:.3f} | {text_preview}...')